<a href="https://colab.research.google.com/github/MohamedAlMasoudi/cs453-ai-project/blob/master/project.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# CS453 - Two-Week Individual Project Workbook

**Name:** [محمد احمد المسعودي]  
**Student ID:** [09211116]  
**Project:** [Campus Delivery ]

Read the two-week student guide first. This workbook supplies only map data and a variant helper, not search or agent solutions. Complete your own explanations and implementations. Save this notebook in your private GitHub repository as project.ipynb and follow docs/github-colab-guide.md. Submit the repository and final commit links in Classroom. The full course project continues after the professor returns.


## 1. Task, PEAS and environment

**Task.** A Campus Delivery Agent moves around a graph map of a university campus and delivers one order from a start location to a customer's building. The agent can move one cell at a time (Up, Down, Left, Right) and must perform a Deliver action once it reaches the destination.

**Inputs.** The campus graph (free nodes, walls/obstacles, and any movement costs), the agent's start cell, and the delivery goal cell.

**Output.** A sequence of legal actions (for example Right, Right, Down, Deliver) that takes the agent from the start to the goal, together with the path cost and the number of expanded nodes.

**Success criteria.** The order is delivered at the correct cell, every action in the plan is legal (no walls, no leaving the map), and the plan is optimal: BFS gives the fewest steps, and A* gives the lowest cost.

### PEAS
- **Performance measure:** Successful delivery, minimum path cost/number of steps, no illegal moves, and few expanded nodes.
- **Environment:** A discrete 2D campus graph with free nodes, obstacles, a start cell and a goal cell.
- **Actuators:** Move Up, Move Down, Move Left, Move Right, and Deliver.
- **Sensors:** The agent's current position, the map layout (obstacles), the goal position, and whether the order has been delivered.

### Environment properties
- **Fully observable:** The agent knows the whole map, its own position and the goal at all times, so no hidden information is needed.
- **Deterministic:** Each action always has the same result in the simulator (a legal move always succeeds, an illegal move is rejected), so there is no randomness.
- **Sequential:** Each move changes the agent's position and affects which moves are possible later, so decisions depend on earlier ones.
- **Static:** The map does not change while the agent plans, so there are no moving people or changing obstacles.
- **Discrete:** The world has a finite set of graph nodes, actions and states, not continuous positions.
- **Single agent:** Only one delivery agent acts, with no other agents competing or cooperating.

### Three scope limits
1. The map is a simple graph, not a real campus: no pedestrians, vehicles, weather or moving obstacles.
2. Only one agent and one order are handled, with no battery, time windows or multiple deliveries.
3. The agent has perfect knowledge and perfect execution: there is no sensor noise, slipping or failed actions.
My assigned variant (student ID 09211116) is variant 1: the goal is F and the road B–E is removed.

## 2. Representation

**State.** A state is a pair `(location, delivered)`: `location` is a node of the campus graph (S, A, B, C, D, E, F, G) and `delivered` is a Boolean task status.

**Initial state.** `("S", False)`. My assigned variant (ID 09211116) is variant 1: the goal is **F** and the road **B–E** is removed.

**Legal actions.**
- `Move(n)`: legal only if `n` is a neighbor of the current location in the assigned graph. It does not change `delivered`.
- `Deliver`: legal only if `location == "F"` and `delivered == False`.

**Result / transition.** `Move(n)` gives `(n, delivered)`. `Deliver` gives `(location, True)`. Any other action is illegal; the simulator rejects it with an error and does not change the state.

**Repeated service.** After `delivered == True`, `Deliver` is illegal, so the order cannot be delivered twice. `Deliver` anywhere except F is also illegal.

**Goal test.** A state is a goal state if `delivered == True`. This can only happen after `Deliver` at F.

**Costs.** Every action costs 1, so path cost equals the number of actions. Search plans the moves; `Deliver` is executed by the simulator at the end.

**Initially complete?** No. The start is S, not F, and `delivered` is False.

In [17]:
# Supplied data/helper: cite CS453 two-week project workbook if adapted.
from copy import deepcopy
import platform

REFERENCE_GRAPH = {
    "S": ["A", "B"],
    "A": ["S", "C", "D"],
    "B": ["S", "E"],
    "C": ["A", "F"],
    "D": ["A", "G"],
    "E": ["B", "G"],
    "F": ["C", "G"],
    "G": ["D", "E", "F"],
}
VARIANTS = [("A", "D", "G"), ("B", "E", "F"),
            ("C", "F", "E"), ("E", "G", "D")]

def make_variant(student_id):
    if not str(student_id).strip():
        raise ValueError("Enter your student ID first")
    variant = sum(ord(ch) for ch in str(student_id).strip()) % 4
    graph = deepcopy(REFERENCE_GRAPH)
    u, v, goal = VARIANTS[variant]
    graph[u].remove(v)
    graph[v].remove(u)
    return {"variant": variant, "graph": graph, "start": "S",
            "goal": goal, "removed_edge": (u, v)}

STUDENT_ID = "09211116"
if STUDENT_ID.strip():
    assigned = make_variant(STUDENT_ID)
    print(assigned)
else:
    print("Unfinished template: enter your ID to generate your variant.")
print("Python:", platform.python_version())


{'variant': 1, 'graph': {'S': ['A', 'B'], 'A': ['S', 'C', 'D'], 'B': ['S'], 'C': ['A', 'F'], 'D': ['A', 'G'], 'E': ['G'], 'F': ['C', 'G'], 'G': ['D', 'E', 'F']}, 'start': 'S', 'goal': 'F', 'removed_edge': ('B', 'E')}
Python: 3.13.18


### Supplied heuristic data for Task 6

These are shortest distances in the reference graph with all original roads open. Your assigned variant removes a road, so those relaxed-world distances cannot overestimate the remaining route cost. They are supplied task data; students implement the actual BFS and A* algorithms. For goal F: S=3, A=2, B=3, C=1, D=2, E=2, F=0, G=1. No extra hand-trace task is required.

In [18]:
# Supplied heuristic data: h is distance in the original graph before a road closure.
# Removing roads cannot make a route shorter, so these values remain lower bounds.
# h(goal) = 0. These tables apply to the original graph and its edge-deletion variants.
# If you add shortcuts/change costs, re-justify h or use h = 0 instead.
HEURISTICS = {'A': {'A': 0, 'B': 2, 'C': 1, 'D': 1, 'E': 3, 'F': 2, 'G': 2, 'S': 1}, 'B': {'A': 2, 'B': 0, 'C': 3, 'D': 3, 'E': 1, 'F': 3, 'G': 2, 'S': 1}, 'C': {'A': 1, 'B': 3, 'C': 0, 'D': 2, 'E': 3, 'F': 1, 'G': 2, 'S': 2}, 'D': {'A': 1, 'B': 3, 'C': 2, 'D': 0, 'E': 2, 'F': 2, 'G': 1, 'S': 2}, 'E': {'A': 3, 'B': 1, 'C': 3, 'D': 2, 'E': 0, 'F': 2, 'G': 1, 'S': 2}, 'F': {'A': 2, 'B': 3, 'C': 1, 'D': 2, 'E': 2, 'F': 0, 'G': 1, 'S': 3}, 'G': {'A': 2, 'B': 2, 'C': 2, 'D': 1, 'E': 1, 'F': 1, 'G': 0, 'S': 3}, 'S': {'A': 1, 'B': 1, 'C': 2, 'D': 2, 'E': 2, 'F': 3, 'G': 3, 'S': 0}}
print("Heuristic tables available for targets:", ", ".join(sorted(HEURISTICS)))


Heuristic tables available for targets: A, B, C, D, E, F, G, S


## 4. Implement and test the simulator

Write functions for legal moves, state updates, service action and task completion. Reject invalid actions clearly. Do not mutate shared map data between tests. Search plans the route; the simulator verifies/exercises its moves and final service.


In [19]:
def initial_state(assigned):
    return (assigned["start"], False)

def legal_actions(state, graph, goal):
    loc, delivered = state
    if loc not in graph:
        raise ValueError(f"Invalid location: {loc}")
    actions = [f"Move({n})" for n in graph[loc]]
    if loc == goal and not delivered:
        actions.append("Deliver")
    return actions

def step(state, action, graph, goal):
    """Return (new_state, cost). Illegal actions raise ValueError. Inputs are not mutated."""
    if action not in legal_actions(state, graph, goal):
        raise ValueError(f"Illegal action {action} in state {state}")
    loc, delivered = state
    if action == "Deliver":
        return (loc, True), 1
    return (action[5:-1], delivered), 1

def is_goal(state):
    return state[1] is True

def run_plan(plan, start_state, graph, goal):
    state, total, trace = start_state, 0, [start_state]
    for action in plan:
        state, cost = step(state, action, graph, goal)
        total += cost
        trace.append(state)
    return state, total, trace

# ---- Simulator tests ----
graph, goal = assigned["graph"], assigned["goal"]
graph_before = deepcopy(graph)
s0 = initial_state(assigned)
print("Initial state:", s0, "| complete?", is_goal(s0))
print("Legal at S:", legal_actions(s0, graph, goal))

tests = [
    ("Move(E) from S (not a neighbor)", s0, "Move(E)"),
    ("Deliver at S (not the goal)", s0, "Deliver"),
    ("Move(B) then Move(E) (removed road)", ("B", False), "Move(E)"),
    ("Deliver twice", ("F", True), "Deliver"),
    ("Invalid location", ("Z", False), "Move(A)"),
]
for name, st, act in tests:
    try:
        step(st, act, graph, goal)
        print("NOT REJECTED:", name)
    except ValueError as e:
        print("Rejected OK ->", name, "|", e)

final, cost, _ = run_plan(["Move(A)", "Move(C)", "Move(F)", "Deliver"], s0, graph, goal)
print("Manual plan final state:", final, "| cost:", cost, "| goal?", is_goal(final))
print("Graph unchanged:", graph == graph_before)

Initial state: ('S', False) | complete? False
Legal at S: ['Move(A)', 'Move(B)']
Rejected OK -> Move(E) from S (not a neighbor) | Illegal action Move(E) in state ('S', False)
Rejected OK -> Deliver at S (not the goal) | Illegal action Deliver in state ('S', False)
Rejected OK -> Move(B) then Move(E) (removed road) | Illegal action Move(E) in state ('B', False)
Rejected OK -> Deliver twice | Illegal action Deliver in state ('F', True)
Rejected OK -> Invalid location | Invalid location: Z
Manual plan final state: ('F', True) | cost: 4 | goal? True
Graph unchanged: True


## 5. Uninformed search: BFS

BFS is uninformed because it uses no knowledge about how close a node is to the goal; it only uses the graph structure. It keeps a FIFO queue, so nodes are expanded in order of their distance (number of moves) from S: all nodes at depth d are expanded before any node at depth d+1. A `parent` dictionary also serves as the visited set, so no location is queued twice, and it is used to reconstruct the route backwards from the goal. On these unit-cost graphs every move costs 1, so the first time BFS reaches the goal it has used the minimum number of moves. Edge cases: if start equals goal, the route is just `[start]`; if no route exists, BFS returns `None`; unknown locations raise a `ValueError`.

In [20]:
from collections import deque

def reconstruct_path(parent, goal):
    path, node = [], goal
    while node is not None:
        path.append(node)
        node = parent[node]
    return path[::-1]

def bfs(graph, start, goal):
    """Return (path or None, number of expanded nodes)."""
    if start not in graph or goal not in graph:
        raise ValueError("Invalid start or goal location")
    if start == goal:
        return [start], 0
    frontier = deque([start])
    parent = {start: None}
    expanded = 0
    while frontier:
        node = frontier.popleft()
        expanded += 1
        for nb in graph[node]:
            if nb not in parent:
                parent[nb] = node
                if nb == goal:
                    return reconstruct_path(parent, goal), expanded
                frontier.append(nb)
    return None, expanded

def path_to_actions(path):
    return [f"Move({n})" for n in path[1:]] + ["Deliver"]

# Edge cases
print("start=goal:", bfs(graph, "S", "S"))
print("no route (E is cut off from S only if...):", bfs({"S": [], "F": []}, "S", "F"))
try:
    bfs(graph, "S", "Z")
except ValueError as e:
    print("invalid location rejected:", e)

# Assigned mission
bfs_path, bfs_expanded = bfs(graph, assigned["start"], goal)
bfs_actions = path_to_actions(bfs_path)
final, total_cost, trace = run_plan(bfs_actions, initial_state(assigned), graph, goal)
print("BFS route:", " -> ".join(bfs_path))
print("Moves:", len(bfs_path) - 1, "| expanded nodes:", bfs_expanded)
print("Actions:", bfs_actions)
print("Execution trace:", trace)
print("Final state:", final, "| total cost:", total_cost, "| task complete?", is_goal(final))

start=goal: (['S'], 0)
no route (E is cut off from S only if...): (None, 1)
invalid location rejected: Invalid start or goal location
BFS route: S -> A -> C -> F
Moves: 3 | expanded nodes: 4
Actions: ['Move(A)', 'Move(C)', 'Move(F)', 'Deliver']
Execution trace: [('S', False), ('A', False), ('C', False), ('F', False), ('F', True)]
Final state: ('F', True) | total cost: 4 | task complete? True


## 6. Informed search: A*

A* expands the node with the smallest f(n) = g(n) + h(n). Here g(n) is the real cost (number of moves) from S to n found so far, and h(n) is the estimated remaining cost from n to the goal F, taken from the supplied `HEURISTICS["F"]` table. The frontier is a priority queue (heap), `best_g` stores the cheapest known cost to each node, and `parent` allows route reconstruction. If a cheaper way to a node is found, its entry is updated; outdated heap entries are skipped. The goal test is done when a node is removed from the frontier (not when it is generated), which is what keeps A* optimal.

The supplied h is admissible because it is the shortest distance in the original graph with all roads open. My variant removes the road B–E, and removing a road can never make a route shorter, so the true remaining cost in my graph is greater than or equal to h; h never overestimates. It would need revising (or be replaced by h = 0) if I added shortcuts, changed road costs, or changed the goal, because then the real cost could fall below h or the table would belong to a different goal.

A* and BFS return the same number of moves (3); the printed paths and expanded-node counts are compared below.

In [21]:
import heapq, itertools

def astar(graph, start, goal, h):
    """Return (path or None, cost or None, number of expanded nodes). Every move costs 1."""
    if start not in graph or goal not in graph:
        raise ValueError("Invalid start or goal location")
    tie = itertools.count()
    frontier = [(h[start], 0, next(tie), start)]
    best_g = {start: 0}
    parent = {start: None}
    expanded = 0
    while frontier:
        f, g, _, node = heapq.heappop(frontier)
        if g > best_g[node]:
            continue                      # outdated entry
        if node == goal:                  # goal test on removal from frontier
            return reconstruct_path(parent, goal), g, expanded
        expanded += 1
        for nb in graph[node]:
            new_g = g + 1
            if nb not in best_g or new_g < best_g[nb]:
                best_g[nb] = new_g
                parent[nb] = node
                heapq.heappush(frontier, (new_g + h[nb], new_g, next(tie), nb))
    return None, None, expanded

h = HEURISTICS[goal]
print("start=goal:", astar(graph, "S", "S", HEURISTICS["S"]))

astar_path, astar_cost, astar_expanded = astar(graph, assigned["start"], goal, h)
astar_actions = path_to_actions(astar_path)
final, total_cost, trace = run_plan(astar_actions, initial_state(assigned), graph, goal)
print("A* route:", " -> ".join(astar_path))
print("Moves (g):", astar_cost, "| expanded nodes:", astar_expanded)
print("Actions:", astar_actions)
print("Execution trace:", trace)
print("Final state:", final, "| total cost:", total_cost, "| task complete?", is_goal(final))

print("Same minimum move count as BFS:", len(astar_path) == len(bfs_path))
print("BFS expanded:", bfs_expanded, "| A* expanded:", astar_expanded)

start=goal: (['S'], 0, 0)
A* route: S -> A -> C -> F
Moves (g): 3 | expanded nodes: 3
Actions: ['Move(A)', 'Move(C)', 'Move(F)', 'Deliver']
Execution trace: [('S', False), ('A', False), ('C', False), ('F', False), ('F', True)]
Final state: ('F', True) | total cost: 4 | task complete? True
Same minimum move count as BFS: True
BFS expanded: 4 | A* expanded: 3


## 10. Sources, assistance and questions

**Sources.** I used the supplied CS453 project workbook data (`REFERENCE_GRAPH`, `make_variant`, `HEURISTICS`) unchanged. I used the course graph-search and informed-search study notebooks as references for the BFS and A* ideas; the code in this notebook was written/adapted for my graph-based Campus Delivery Agent with a `(location, delivered)` state.

**AI assistance.** I used Claude (Anthropic) to help understand the assignment, draft the text for Tasks 1, 2, 5, 6 and 10, and draft the simulator, BFS and A* code. I verified the work by running all cells, checking that the simulator rejects illegal actions, hand-tracing the route S → A → C → F for my variant, and confirming that BFS and A* return the same number of moves.

**Unresolved questions.** None.

## GitHub + Colab workflow

1. On the public starter, use Use this template > Create a new repository, choose Private, then invite the professor using the announced GitHub username.
2. Your copy already includes project.ipynb. Open it in Colab using authorized GitHub access or download/upload.
3. Save each work session explicitly to GitHub: File > Save a copy in/to GitHub, correct repo/branch and project.ipynb. Drive autosave alone does not update GitHub.
4. If integration fails, download the notebook from Colab, upload it to GitHub to replace project.ipynb, and commit. No terminal or embedded tokens are needed.
5. Commit real progress throughout the project. Complete README.md using the supplied template.
6. Reopen the committed notebook in a fresh Colab session and verify it runs. Submit repo URL + final commit URL in Classroom and Turn in.

See the student guide pages 5-6 and docs/github-colab-guide.md for details. The professor must accept the invitation before viewing private work; do not wait for acceptance to begin.


## 11. Submission checklist

- [x] Identity, chosen project and generated variant recorded
- [x] Task 1: PEAS and environment explained
- [x] Task 2: state, legal actions, transition, goal and costs defined
- [x] Task 4: legal movement, service and task goal implemented
- [x] Task 5: working BFS and assigned mission output
- [x] Task 6: working A*, admissible heuristic explanation and assigned mission output
- [x] Sources and assistance disclosed
- [x] README with task/variant/files/dependencies/run steps
- [x] Clean restart/run-all checked and saved
- [x] project.ipynb and README.md saved in my private GitHub repository
- [ ] Professor invited; repository URL and final commit URL submitted in Classroom

Original task numbers are retained for reference. Tasks 3, 7, 8 and 9 are removed. There is no Day 7 checkpoint.